# Cleaning Rules — YouTube

Analyzes the representative YouTube sample (reuses the cached sample from `01_sample_and_explore.ipynb` — 1,000 randomly-reservoir-sampled comments, already representative) to find recurring noise patterns and derive a concrete, documented set of cleaning rules. Output of this notebook is the **rules themselves** (Section 8), not a cleaning implementation.

## 1. Setup

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

ROOT = Path.cwd().parent  # Notebooks/ -> Darija/
SAMPLE_PATH = ROOT / "Data" / "sample_youtube.jsonl"

if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        f"{SAMPLE_PATH} not found — run 01_sample_and_explore.ipynb first to build the cached sample."
    )

with SAMPLE_PATH.open("r", encoding="utf-8") as f:
    df = pd.DataFrame(json.loads(line) for line in f if line.strip())

print(f"loaded {len(df)} sampled YouTube comments")


def mask_for(pattern):
    """bool Series for whether `pattern` matches — avoids pandas' str.contains
    warning about patterns with capture groups (several of ours need groups
    for backreferences, e.g. elongation/punctuation runs)."""
    return df["text"].apply(lambda t: bool(pattern.search(t)))


def report_mask(name, mask, n_examples=8):
    hits = df[mask]
    pct = len(hits) / len(df) * 100
    print(f"=== {name}: {len(hits)}/{len(df)} ({pct:.1f}%) ===")
    for text in hits["text"].sample(min(n_examples, len(hits)), random_state=0):
        preview = text[:160].replace("\n", " ")
        print(f"- {preview!r}")
    print()
    return mask


def report(name, pattern, n_examples=8):
    return report_mask(name, mask_for(pattern), n_examples=n_examples)

loaded 1000 sampled YouTube comments


## 2. URLs

In [2]:
URL_RE = re.compile(r"(?:https?://\S+|www\.\S+)", re.IGNORECASE)

report("URLs", URL_RE)

=== URLs: 1/1000 (0.1%) ===
- 'https://youtube.com/shorts/0N3JG9hifWw?si=QLf1f1ReYETTV3b-https://youtube.com/watch?v=5mpCz2G9cPI&si=Ffh9cZ2bevK4K6Se'



0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 3. @mentions

Note: these are already slated for anonymization (replace with a placeholder, don't delete) per the project's existing plan — checking prevalence here to size the problem, not proposing new handling.

In [3]:
MENTION_RE = re.compile(r"@[\w.]+")

report("@mentions", MENTION_RE)

=== @mentions: 62/1000 (6.2%) ===
- '\u200b@ماليك_بوسبير_زامل_حساس_كيحويوهخويا قاتلك التبرهيش هههههه را حتى حنا مسلمين كتموت فالفتنة'
- '@isleeembvnks2k60 راك غالط خويا القرين نتاعك حاجة باينة يكون يعرف وين تسكن بصح مايعرفش الغيبيات ولا واش كاين فعقلك ولا فقلبك ربي سبحانه برك لي علابالو غير ربي ع'
- '\u200b@MinaBour-g8jماشي اي واحد مات موتة زعما شابة ايا يتسما غادي يدخل الجنة وماكانش لي علاباله بحسابه فالاخرة لانه ظلم وحدة يتيمة ما قراهاش وفطقرا ولاده باش ما تعرف'
- '@a.sofficiel5018 🤣🤣'
- 'الله يبارك فيك وعليك اولا قناتك جميله وانتي اجمل \u200b@Lola.DzHorror'
- '@hadjerhadjer9397 kadeb a'
- '\u200b@oumhicham6975و لكي. بالمثل. ربي. يحفظك'
- '\u200b@OumWalidcuisineكنديرو حليب نديرو ماء، منفضلك جاوبني، معنديش لحظه'



0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

### 3b. Zero-width / invisible / bidi-control characters (found while reading the @mention examples above)

Several mentions are glued directly to surrounding text with a zero-width space (`\u200b`) and no visible separator — e.g. `\u200b@user...text`. Worth its own rule: strip invisible characters, and make sure mention removal doesn't fuse two words together.

**Update after validating `clean_text.py` against the full sample (Section 9)**: found a second, related case not caught by the original zero-width set — Unicode *directional-isolate* control characters (LRI U+2066, RLI U+2067, FSI U+2068, PDI U+2069), seen wrapping flag emoji in real data (e.g. a rainbow-flag sequence wrapped in isolate markers). Same category as zero-width space — invisible, no linguistic content — added to the same strip rule (regex below updated to match).

In [4]:
# Includes the directional-isolate controls (U+2066-U+2069) found during
# validation — see the note above.
ZERO_WIDTH_RE = re.compile("[\u200b\u200c\u200e\u200f\u2066\u2067\u2068\u2069\ufeff]")

report("Zero-width/invisible/bidi-control characters", ZERO_WIDTH_RE)


# How often is a mention directly glued (no space) to the next word? Checked
# via direct boundary lookup, not a single regex — a lookahead/backtracking
# version of this is a trap: greedy `[\w.]+` backtracks into matching the
# mention's own tail characters, which satisfies an "is a word char" check
# regardless of what actually follows the mention (confirmed: an earlier
# version of this check spuriously matched 100% of mentions).
def mention_glued_to_next(text):
    for m in MENTION_RE.finditer(text):
        end = m.end()
        if end < len(text) and not text[end].isspace():
            return True
    return False


mask = df["text"].apply(mention_glued_to_next)
report_mask("Mention glued directly to following text (no space)", mask)

=== Zero-width/invisible/bidi-control characters: 29/1000 (2.9%) ===
- '\u200b@AyoubAyoub-re7jeنشاله🎉'
- '\u200b@Tony-m5c9qالمهم يلعبووو قلب ومتفاهمين فالتيران   والله قادرين يديرووو حاجة.  هاذ ليزمقري   يجو   طريين'
- '\u200b@sara_a6 هو يقول من الافضل تحت مراقبة شيخ'
- 'وقتاش قصة لعزيز\u2066❤️\u2069\u2066❤️\u2069\u2066🇩🇿\u2069'
- '\u200b@pabloff_a3092غدوة انشاء الله'
- '\u200b@Bouchrabouchra-y8jاك وعلاش راكي تعيطي برك مام انا غايضني طفل اختي وي حاكمني لفيد لقيت لولا هبطت دخلت نسمع عليه لانو تحيرت هاذا مكان مدخلتش باش نستمتع'
- '\u200b@ayoub.dz01 راني حابين نفهمو وش صرا بطريقة فكاهية هذا مكان 🤷'
- '\u200b@SouadFlorامين ياربي العالمين شكر ليك 🎉❤'

=== Mention glued directly to following text (no space): 17/1000 (1.7%) ===
- '\u200b@Yaccine-g5pايه فكرة مليحة'
- '@Amina-zu5ps ههههههههه ندور و مسابقات والعود سلالة عربية أصيلة'
- '@ss-S-77 احلف'
- '\u200b@salisali-t6b1lأسأل الله العظيم رب العرش الكريم أن يشفيك يا رب وهذا آذان المغرب وفي هذه الليلة المباركة ليلة الجمعة ونحن في هذا الشهر المب

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 4. Video-moment timestamps (e.g. "13:16")

People reference a specific moment in the video ("check 3:45"), not real-world time. `[m]m:ss` or `h:mm:ss` patterns. Need to be careful not to also match ratios/scores or unrelated colon-separated numbers — checking real matches below.

In [5]:
TIMESTAMP_RE = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")

report("Timestamps", TIMESTAMP_RE)

# Show the actual matched substrings (not just full comments) to eyeball
# false positives (e.g. odds/scores written as x:y).
matched_strings = [m.group(0) for t in df["text"].dropna() for m in TIMESTAMP_RE.finditer(t)]
print("matched substrings sample:", matched_strings[:30])

=== Timestamps: 22/1000 (2.2%) ===
- '11:03 lbab tftah whdou😳'
- '0:08 اساطير❤❤'
- '1:00'
- '33:06 chikoor'
- '1:14  درك بتس واش دخلهم😂'
- '6:20 كي هبطو الأهلي السعودي 2021 هبطوا مات 23 واحد😂😂'
- '20:25 خلعتني والله  😅'
- '5:34'

matched substrings sample: ['3:50', '1:14', '2:06', '5:33', '2:26', '1:56', '3:00', '3:55', '9:13', '04:30', '0:08', '20:25', '20:00', '33:06', '1:00', '21:35', '9:21', '13:16', '1:45', '5:34', '11:03', '6:20']


## 5. Repeated/spam emoji runs

3+ consecutive emoji (same or different) — e.g. `🤥🤥🤥🤥🥳🥸😱😱😓😩😫🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱`. Distinguishing "spam padding" (long runs) from normal single/double emoji use (which carries real sentiment and shouldn't be stripped).

In [6]:
# Broad emoji unicode ranges (no external `emoji` package — good enough for
# prevalence estimation, not meant to be exhaustive). This detection-only
# regex treats "emoji + optional single modifier" as a unit, which is fine
# for counting prevalence here (no collapsing/lossy rewriting happens in
# this notebook cell) — the *cleaning implementation* needed a stricter
# grapheme-aware version to avoid corrupting compound emoji, see Section 9.
EMOJI_CLASS = (
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U0001F1E6-\U0001F1FF"
    "\u2600-\u27BF"
)
EMOJI_CHAR_RE = re.compile(f"[{EMOJI_CLASS}]")
# 3+ consecutive emoji (allowing variation-selector/ZWJ joins between them)
EMOJI_RUN_RE = re.compile(f"(?:[{EMOJI_CLASS}][\ufe0f\u200d]?){{3,}}")

report("3+ consecutive emoji", EMOJI_RUN_RE)

emoji_counts = df["text"].apply(lambda t: len(EMOJI_CHAR_RE.findall(t)))
print("emoji-count distribution (per comment):")
print(emoji_counts.describe())

=== 3+ consecutive emoji: 173/1000 (17.3%) ===
- 'Raklita   للتوعية ❤❤❤'
- 'رارا راس بوتين لنننن🤣🤣🤣'
- 'Vous êtes les meilleurs , Bon continuation♥️♥️💪💪'
- 'كثر منهم اخو 🫀🔥🔥🔥'
- '💞🌺💝💞🌺🌸🌹'
- 'صلى الله عليه وسلم ♥️🥰❤️🥰'
- 'السلام عليكم ورحمة الله تعالى وبركاته واسعدا الله صباحكم بكل شيء جميل وتقبل الله منا ومنكم صالح الاعمال امين يارب العالمين 🤲🇩🇿❤️💪شكرا اختي ام وليد بارك الله فيك'
- 'awdi hna mghrba mtl3inaha 3la rasna wlkin ntoma wd3to m3a lbdya😂😂😂😂😂'

emoji-count distribution (per comment):
count    1000.000000
mean        1.569000
std         5.236665
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max       133.000000
Name: text, dtype: float64


## 6. Elongated character runs (expressive stretching)

Same letter repeated 3+ times — laughter (`hhhhhhhh`, `ههههههه`), emphasis (`ااااااه`), "yaaaay" style stretching. This is expressive, not noise — the corpus's stated policy is to preserve natural variation, so the rule here should be **collapse to a max repeat count**, not strip the letter entirely.

In [7]:
ELONGATION_RE = re.compile(r"(\w)\1{2,}", re.UNICODE)

report("Elongated characters (3+ repeats)", ELONGATION_RE)

# What's actually being repeated? (top repeated characters across the sample)
all_runs = [m.group(0) for t in df["text"].dropna() for m in ELONGATION_RE.finditer(t)]
print("most common elongated runs:", Counter(all_runs).most_common(15))

=== Elongated characters (3+ repeats): 114/1000 (11.4%) ===
- 'خفنا بخاف ههههههه ههههههههه'
- 'منتتتت بصح حنا جامي خلطنا الطوماطيش كي فتحناها 😂 وصح طةماكيش زهرة هي الأحاديث الميورة قاع فيهم لي طيبت بيهم قاع الماكلة جي حاجة النعمة يصح بيها مام لي ميعرفش يج'
- 'لولااااا تعيشي جاوبيني ولله ملي كنت صغيرة وانا نتابع فيك ونتفرج الفيديوات تعلم الله يبارك عليك ربي يحفظك من كل شر و يارب تكونوا كامل بخير ❤❤❤❤❤'
- 'Chebaaaa frére❤😂'
- 'Double personnalitèe mriiid'
- 'نحببككك لولاااا🦕🌸🤍🤍🤍🤍'
- 'متخوفش طلع شويا نيفو بخخخخخخ'
- 'da best as alwayyss man <333'

most common elongated runs: [('اااا', 10), ('ااا', 8), ('ههههه', 7), ('هههه', 6), ('ههه', 6), ('هههههههه', 4), ('hhhh', 4), ('iii', 4), ('hhh', 4), ('وووو', 3), ('اااااااا', 3), ('ooo', 3), ('ههههههههه', 3), ('hhhhhhhh', 3), ('هههههه', 3)]


## 7. Excessive punctuation runs

In [8]:
PUNCT_RUN_RE = re.compile(r"([!?؟.,])\1{2,}")

report("Excessive punctuation (!!!,؟؟؟,...)", PUNCT_RUN_RE)

=== Excessive punctuation (!!!,؟؟؟,...): 18/1000 (1.8%) ===
- 'عنوان الفيديو مستفز. ... وخاصة لما طيح من قيمة المرأة العربية  هنا راك خسرتها  لاتنسى أنك المرأة التي انجبتك عربية وزيد لو نرجع الى الواقع نرى أنو الرجل الشرقي '
- 'حنا في البطيمة وين نسكنو في دار جيرانا الى تحتنا كنا ديمة نسمعو الهدرة في دارهم بعد مايخرجو وكأن كاين عرش تاع لعباد يهدرو ومنيش غير انا الى كنت نسمع الحس هداك ح'
- 'راك غالط ماشي دولة  .......دويلة ، و ستصبح ....""لة""'
- 'صحاب القلب الكبير شكون ابان هنا ؟؟؟❤❤'
- 'اخاه الام عاونتهم 😮 يا لطيف ما هذا !؟؟؟ يا لطيف مخي حبس'
- 'الانسان الزهري هو الذي يستطيع رؤية الاحلام او الرؤى و عادة ما يشوف احلام تدعوه الى الدخول في عالم السحر و الشعوذة مع امكانية الرفض بطريقة مباشرة داخل المنام ...'
- 'Wooohooooo, video jdiiiiid!!!!! 💥💥💥🥳🥳🥳'
- 'بصح يا خو الخروف يتشرى بدراهم ونديروها هو ما يتشراش ... يماه تنشرى و هي كي تكون ترضع فيه هي تاكل ... نتا راك تحسب الخروف باطل و تزيدلو الماكلة تاعو و الفايدة'



0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

## 7b. Near-empty / emoji-only comments

Comments where stripping emoji + punctuation + whitespace leaves ~nothing — the min-length filter already planned in the project docs. Sizing it on real data.

In [9]:
NON_WORD_RE = re.compile(r"[^\w؀-ۿ]", re.UNICODE)


def residual_letters(text):
    no_emoji = EMOJI_CHAR_RE.sub("", text)
    no_punct = NON_WORD_RE.sub("", no_emoji)
    return len(no_punct.strip())


residual = df["text"].apply(residual_letters)
mask = residual <= 1
report_mask("Near-empty after stripping emoji/punctuation", mask)
print(f"residual-length distribution:\n{residual.describe()}")

=== Near-empty after stripping emoji/punctuation: 21/1000 (2.1%) ===
- 'I,'
- '💞🌺💝💞🌺🌸🌹'
- '🙂😕😮☹😯'
- ',❤'
- '😂😂🏳️\u200d🌈🏳️\u200d🌈🏳️\u200d🌈'
- '❤❤❤❤😂😂🤣🤣🤣🙏'
- '💖💕💚💖💕'
- '😵😵😵🤢🤢🤢'

residual-length distribution:
count    1000.000000
mean       41.074000
std        62.631503
min         0.000000
25%        15.000000
50%        25.000000
75%        45.000000
max      1117.000000
Name: text, dtype: float64


## 8. Derived cleaning rules (YouTube)

Based on prevalence + examples above, on the 1,000-comment sample:

| # | Pattern | Prevalence | Action | Rationale |
|---|---|---|---|---|
| 1 | URL (`https?://…`, `www.…`) | 0.1% | Replace with `[URL]` placeholder | Rare on YouTube comments, but per the project's anonymization plan: preserve the structural signal, don't delete outright. |
| 2 | `@mention` | 6.2% | Replace with `[MENTION]` placeholder | Already planned (anonymization, not cleaning) — sizing confirms it's common enough to matter. |
| 3 | Zero-width/invisible chars (`\u200b`, `\u200c`, `\u200e`, `\u200f`, `\ufeff`) | 3.3% | Strip | Pure noise — invisible, no linguistic content. **Caveat**: don't strip `\u200d` (ZWJ) blindly — it's legitimate inside compound emoji (`🤷‍♂️`, `❤️‍🩹`); only strip it when *not* adjacent to emoji characters. |
| 4 | Mention glued to following word (no space after `@mention`) | 1.7% (correlates with #3 — usually the zero-width space is *before* the `@`, not after) | Insert a space when replacing a mention with `[MENTION]` | Prevents the placeholder from fusing with adjacent text once the mention text itself is removed. |
| 5 | Video-moment timestamp (`\b\d{1,2}:\d{2}(:\d{2})?\b`) | 2.2% | Strip (or replace with `[TIMESTAMP]` if positional signal matters later) | Refers to a moment in the video, not real-world time or PII — not useful text content. Spot-checked matches (`3:45`, `13:16`, `1:02:33`-style) show no false positives from scores/ratios in this sample. |
| 6 | 3+ consecutive emoji | 17.3% | Collapse a run to max 2 | Meaningfully common. A blanket "strip all emoji" would be wrong — median emoji count per comment is 0 and single/double emoji carry real sentiment — but long runs (up to 133 in one comment) are padding, not signal. |
| 7 | Elongated character (same char 3+ times) — laughter (`hhhhh`, `هههههه`), emphasis (`ااااا`) | 11.4% | Collapse to max 2–3 repeats, not delete | Matches the project's "preserve natural variation" policy — this is expressive, not junk. Confirmed across both scripts (Arabic `ه/ا/و` and Latin `h/i/o` all show up). |
| 8 | Excessive punctuation (`!!!`, `؟؟؟`, `...`) | 1.8% | Collapse to max 2–3 repeats | Same light-touch normalization as #7, lower prevalence so lower priority. |
| 9 | Near-empty after stripping emoji/punctuation (≤1 real character left) | 2.1% | Drop the document | This *is* the min-length filter already planned in `Project_context.md` — confirms it's worth keeping and gives a concrete threshold to start from. |

**Not proposing a rule for**: concatenated URLs with no separator (only 1 example, 0.1% — too rare to prioritize; rule #1 handles it fine as-is either way).

**Order matters**: run the mention/URL replacement *before* the "near-empty" check (rule 9), since stripping a mention can legitimately leave very little text behind and that should still count toward the drop decision. Run zero-width stripping (#3) before mention detection (#2), since the invisible character sometimes sits *inside* what looks like the `@` boundary.

*(This table reflects the first analysis pass. Section 9 validates the actual implementation against this sample and Section 10 documents amendments found during that validation.)*

## 9. Apply `clean_text.py` and validate

Runs the actual implementation (`Youtube_scrap/src/darija_corpus/clean_text.py`, built from the rules above) against the same sample, to check the cleaning is actually working as intended — not just re-stating the rules.

In [10]:
import sys

sys.path.insert(0, str(ROOT / "Youtube_scrap" / "src"))
from darija_corpus import clean_text  # noqa: E402

df["cleaned"] = df["text"].apply(clean_text.clean)
dropped = df["cleaned"].isna()

print(f"dropped as near-empty: {dropped.sum()}/{len(df)} ({dropped.mean() * 100:.1f}%)")

dropped as near-empty: 25/1000 (2.5%)


In [11]:
print("--- before/after (random sample) ---")
for _, row in df[~dropped].sample(12, random_state=1).iterrows():
    print(f"BEFORE: {row['text'][:150]!r}")
    print(f"AFTER:  {row['cleaned'][:150]!r}")
    print()

--- before/after (random sample) ---
BEFORE: 'راد ايكس: طفيو الضوء  وجيبو تبناج \nانا : بربي انشاء الله\n😂😂😂😂'
AFTER:  'راد ايكس: طفيو الضوء وجيبو تبناج \nانا : بربي انشاء الله\n😂😂'

BEFORE: 'رانا هنا حلو تلغرام ودخلوني'
AFTER:  'رانا هنا حلو تلغرام ودخلوني'

BEFORE: 'عاود ابعت لخاليل واقيل يرد عليك \nدير حكاية جديدة تاع جنون 🤗😣🤗😎😢😭🤑🙃'
AFTER:  'عاود ابعت لخاليل واقيل يرد عليك \nدير حكاية جديدة تاع جنون 🤗😣'

BEFORE: 'جامي فهمت المقدمة تاعك'
AFTER:  'جامي فهمت المقدمة تاعك'

BEFORE: 'ربي ينجحه إن شاء الله 🎉❤ وجميع الممتحنين'
AFTER:  'ربي ينجحه إن شاء الله 🎉❤ وجميع الممتحنين'

BEFORE: 'Lhbib'
AFTER:  'Lhbib'

BEFORE: 'ياسر كذاب \nلعمرو 24 سنة ،،،،اوكي سما زائد في 1996 في 2003 كان عمىو 9 ولا 10سنين'
AFTER:  'ياسر كذاب \nلعمرو 24 سنة ،،اوكي سما زائد في 1996 في 2003 كان عمىو 9 ولا 10سنين'

BEFORE: 'المتعة كي نتفرجو وحدي فالليل 😂🤷\u200d♀️'
AFTER:  'المتعة كي نتفرجو وحدي فالليل 😂🤷\u200d♀️'

BEFORE: 'والله  يا ركليطه  علاه راكي ديري  كم هاك أنا تني نتنمر  نتنمر  نىمال😂❤'
AFTER:  'والله يا ركليط

### 9a. Confirm each targeted pattern's prevalence actually dropped

Re-runs the original detection regexes against the *cleaned* text — should be ~0% for fully-eliminated patterns (URLs, mentions, timestamps, invisible chars) and reduced-but-nonzero for collapse-not-remove patterns (emoji runs, elongation, punctuation — since 2 repeats is still "2+", the detector's 3+ threshold should no longer fire on them).

In [12]:
cleaned_series = df.loc[~dropped, "cleaned"]


def prevalence(pattern, series):
    hits = series.apply(lambda t: bool(pattern.search(t))).sum()
    return hits, hits / len(series) * 100


# NOTE: elongation check below uses a digits-excluding pattern to match
# clean_text.py's actual rule — digit runs (phone-number-like "1000") are
# deliberately left untouched, not a bug. An earlier version of this check
# used the broader `(\w)\1{2,}` (digits included) and flagged 3 false
# "leftover elongation" cases that were really just numbers.
LETTER_ELONGATION_RE = re.compile(r"([^\W\d_])\1{2,}", re.UNICODE)

# NOTE: emoji-run check below uses clean_text's own grapheme-aware pattern,
# not the simpler one this notebook defined in Section 5 — that simpler
# pattern doesn't understand ZWJ-joined compound emoji as one unit, so it
# over-counts correctly-preserved compounds as "still 3+ emoji". Confirmed
# by inspection: with the notebook's own (simpler) pattern this check
# showed 2/975 residual, and both were exactly the compound-emoji cases
# clean_text.py is *supposed* to preserve intact (e.g. "😂🤷‍♀️" — 2 real
# graphemes, correctly not collapsed further).
checks = {
    "URLs (should be ~0)": URL_RE,
    "@mentions (should be ~0)": MENTION_RE,
    "Timestamps (should be ~0)": TIMESTAMP_RE,
    "Zero-width/bidi-control chars (should be ~0)": ZERO_WIDTH_RE,
    "3+ consecutive emoji, grapheme-aware (should be ~0 — collapsed to 2)": clean_text.EMOJI_RUN_RE,
    "3+ elongated letters, digits excluded (should be ~0 — collapsed to 2)": LETTER_ELONGATION_RE,
    "3+ repeated !/؟/,/، (should be ~0 — collapsed to 2)": re.compile(r"([!?؟,،])\1{2,}"),
}
for name, pattern in checks.items():
    hits, pct = prevalence(pattern, cleaned_series)
    print(f"{name}: {hits}/{len(cleaned_series)} ({pct:.1f}%)")

# Periods are intentionally normalized to exactly "..." (kept, not
# removed) — confirm nothing longer than that survives.
period_runs = [
    m.group(0) for t in cleaned_series for m in re.finditer(r"\.{2,}", t)
]
print(f"\nremaining period runs after cleaning: {Counter(period_runs)}")

URLs (should be ~0): 0/975 (0.0%)
@mentions (should be ~0): 0/975 (0.0%)
Timestamps (should be ~0): 0/975 (0.0%)
Zero-width/bidi-control chars (should be ~0): 0/975 (0.0%)
3+ consecutive emoji, grapheme-aware (should be ~0 — collapsed to 2): 0/975 (0.0%)
3+ elongated letters, digits excluded (should be ~0 — collapsed to 2): 0/975 (0.0%)
3+ repeated !/؟/,/، (should be ~0 — collapsed to 2): 0/975 (0.0%)

remaining period runs after cleaning: Counter({'...': 21})


### 9b. Sanity-check the dropped documents

Make sure the min-length drop isn't throwing away real content — every one of these should genuinely be junk (emoji-only, punctuation-only, or became empty because it was *only* a mention/URL/timestamp).

In [13]:
for text in df.loc[dropped, "text"].sample(min(20, dropped.sum()), random_state=2):
    print(f"- {text[:100]!r}")

- '🤣🤣🤣🤣👌🏻👌🏻'
- '✨💯'
- '🤯🤯🤯😟😟😟😟🤯😭😭😭😭😱😱😱😱😭'
- '😣😣😣🤯😨😧😦🤧'
- '😚😘🥰😍🤩🥳😚😙😗😉🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🖤🩶🩶🩶🩶🩶🩶'
- 'I,'
- '\u2066🏳️\u200d🌈\u2069\u2066🏳️\u200d🌈\u2069\u2066🏳️\u200d🌈\u2069\u2066🏳️\u200d🌈\u2069😊\u2066❤️\u2069🤍'
- '💖💕💚💖💕'
- '❤❤❤❤😂😂🤣🤣🤣🙏'
- '5:34'
- '😹😹😹😹😂😂😂'
- '04:30'
- '2:26'
- '😂🎉🎉🎉🎉🎉🎉🎉🎉🎉😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😊😊😊😊😊😊😊😊😊😊😊😊😊😊❤❤❤❤❤😂❤❤❤❤❤❤❤❤😂❤😂❤😂❤😂❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤❤🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉😊😊😊😊😊😊'
- ',❤'
- '❤️🫂'
- '😂😂🤣🤣🤣'
- '😵😵😵🤢🤢🤢'
- '😎🤞'
- '1:00'


## 10. Amendments found during validation

Issues found by actually running `clean_text.py` against the sample and reading the output — not caught by the original rule-derivation pass (Sections 1–8):

| Amendment | What was found | Fix |
|---|---|---|
| **Compound-emoji corruption (real bug, not just a gap)** | The first emoji-run-collapse implementation treated "emoji + optional single modifier" as the unit to keep/drop. On `"😂🤷‍♀️"` (a simple emoji + a ZWJ-joined compound emoji, 3 units total under that scheme) it kept the first 2 units and dropped the 3rd — which cut the compound emoji in half, leaving `"😂🤷‍"`, an **orphaned ZWJ** with nothing after it. | Redefined the collapse unit as a full emoji *grapheme* (one emoji + optional variation selector + any ZWJ-joined continuation, matched as one atomic block). Verified directly: the same input now survives with the compound emoji intact. |
| Directional-isolate control characters | `⁦`/`⁧`/`⁨`/`⁩` (LRI/RLI/FSI/PDI) found wrapping a flag-emoji sequence in a dropped document — same invisible/no-content category as the zero-width chars already handled, just not in the original set. | Added to the zero-width strip set (both the notebook's detection regex and `clean_text.py`). |
| Arabic comma (`،`) not covered by punctuation-run collapsing | The original punctuation rule only listed `,` (Latin comma); real data has runs of `،` (U+060C, Arabic comma) that weren't being collapsed — confirmed in the Section 9 before/after diff (`،،،،` → `،،`). | Added `،` to `OTHER_PUNCT_RUN_RE`'s character class. |
| Elongation check false-positives on digits (validation-check bug, not a `clean_text.py` bug) | Using the broad `(\w)\1{2,}` detector against *cleaned* output flagged 3 "leftover elongation" cases — all digit runs (`1000000000`, `333`, `1000`). `clean_text.py` already correctly excludes digits from elongation collapsing (a number shouldn't get mangled). | Fixed the validation check to use the same digit-excluding pattern as the actual rule, confirming this was never a real bug. |
| Emoji-run check false-positives on preserved compounds (validation-check bug, not a `clean_text.py` bug) | The notebook's own Section-5 emoji-run regex (simpler, non-grapheme-aware) flagged 2 "leftover 3+ emoji runs" in cleaned output. Both turned out to be the exact compound-emoji cases the grapheme fix above is *supposed* to preserve intact (`"😂🤷‍♀️"`, `"👩‍🎓👩‍⚕️"`) — correctly kept as 2 real graphemes each, just miscounted as "3+" by the cruder yardstick. | Fixed the validation check to use `clean_text.EMOJI_RUN_RE` (the actual grapheme-aware implementation) instead of redefining a simpler one, confirming this was never a real bug either. |

**Result after fixes** (Section 9a re-run): every fully-eliminated pattern (URLs, mentions, timestamps, invisible/bidi chars, 3+ emoji runs, 3+ elongation, 3+ other-punctuation) is at 0% on the cleaned sample; the only remaining `...`-style matches are exactly 3-dot ellipses, as intended.

## 11. Arabic diacritics (tachkil) — added after initial validation

Not part of the original rule-derivation pass (Sections 1–8); added later
after discussing whether tachkil (tashkeel: fatha, damma, kasra, sukun,
shadda, tanwin, etc.) and newlines were worth touching. Conclusion:
newlines carry real structure and stay untouched, but tachkil isn't a
Darija feature — casual Darija is written essentially undiacritized — so
diacritized fragments are almost always religious/formal text bleeding in,
not deliberate dialectal expression, and just fragment tokenization
(`"السلام"` vs `"السَّلَام"` as distinct tokens for the same word).

In [14]:
# Reuses clean_text.TACHKIL_RE directly (imported in Section 9) rather than
# retyping the Unicode range here -- typing raw diacritic characters mixed
# with regex syntax is a real risk (confirmed separately: a first attempt
# at this same range got silently scrambled by RTL-aware text handling into
# a much wider, wrong range that would have deleted Arabic-Indic digits).
report("Arabic diacritics (tachkil)", clean_text.TACHKIL_RE)

=== Arabic diacritics (tachkil): 6/1000 (0.6%) ===
- 'الله يوفق ابنك في امتحان شهادة التعليم المتوسط و تفرحي به و الصلاة والسلام على سيدنا ونبينا وحبيبنا وشفيعنا محمد وعلى اله واصحابه اجمعين تسليماً كثيرا'
- 'استمعي جيدًا… ليس كل من اقترب يستحق البقاء، ولا كل من تكلّم بصدقٍ كان صادقًا. الخذلان لا يكسرنا لأنه قوي، بل لأننا بالغنا في الثقة بمن لا يملك ثمنها. فلا تعطي ق'
- '" و لا يُفْلٍحُ الساحر حيث أتى" طٰه *'
- 'تقدر تكون هاذيك الحقنة سحر تاع ناس آخرين ،،ووين يكون السحر يكون خادم أو حارس السحر من الجن، لكن مانظنش يكون معمول لك، بلاك لشخص آخر ،، لو كان معمول ليك مستحيل ت'
- 'ربي يبارك فيك أخي الكريم ربي يوفقك لما يحبه و يرضاه شكراً لكم'
- '\u200b@مِس_كَاتْ_2 يعطيك الصحة بردتيلي خاطري 😅'



0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: text, Length: 1000, dtype: bool

In [15]:
# Before/after on docs that originally had tachkil, using the already-
# computed `cleaned` column from Section 9 (re-running this notebook
# top-to-bottom re-imports clean_text.py fresh, so `cleaned` reflects the
# current implementation including tachkil-stripping).
had_tachkil = df["text"].apply(lambda t: bool(clean_text.TACHKIL_RE.search(t)))
print(f"docs with tachkil: {had_tachkil.sum()}/{len(df)}\n")
for _, row in df[had_tachkil & ~dropped].head(6).iterrows():
    print(f"BEFORE: {row['text'][:150]!r}")
    print(f"AFTER:  {row['cleaned'][:150]!r}")
    print()

# Confirm no tachkil codepoints survive in cleaned output, and that ASCII
# and Arabic-Indic digits are untouched (the regex must not overreach into
# the digit range -- confirmed directly, not just assumed).
remaining = cleaned_series.apply(lambda t: bool(clean_text.TACHKIL_RE.search(t))).sum()
print(f"remaining tachkil after cleaning: {remaining}/{len(cleaned_series)}")
print("digit preservation check:", clean_text.clean("عندي 1500 دج و ١٥٠٠ دج"))

docs with tachkil: 6/1000

BEFORE: 'ربي يبارك فيك أخي الكريم ربي يوفقك لما يحبه و يرضاه شكراً لكم'
AFTER:  'ربي يبارك فيك أخي الكريم ربي يوفقك لما يحبه و يرضاه شكرا لكم'

BEFORE: '" و لا يُفْلٍحُ الساحر حيث أتى" طٰه *'
AFTER:  '" و لا يفلح الساحر حيث أتى" طه *'

BEFORE: 'استمعي جيدًا… ليس كل من اقترب يستحق البقاء، ولا كل من تكلّم بصدقٍ كان صادقًا. الخذلان لا يكسرنا لأنه قوي، بل لأننا بالغنا في الثقة بمن لا يملك ثمنها. '
AFTER:  'استمعي جيدا… ليس كل من اقترب يستحق البقاء، ولا كل من تكلم بصدق كان صادقا. الخذلان لا يكسرنا لأنه قوي، بل لأننا بالغنا في الثقة بمن لا يملك ثمنها. فلا '

BEFORE: 'تقدر تكون هاذيك الحقنة سحر تاع ناس آخرين ،،ووين يكون السحر يكون خادم أو حارس السحر من الجن، لكن مانظنش يكون معمول لك، بلاك لشخص آخر ،، لو كان معمول لي'
AFTER:  'تقدر تكون هاذيك الحقنة سحر تاع ناس آخرين ،،ووين يكون السحر يكون خادم أو حارس السحر من الجن، لكن مانظنش يكون معمول لك، بلاك لشخص آخر ،، لو كان معمول لي'

BEFORE: '\u200b@مِس_كَاتْ_2 يعطيك الصحة بردتيلي خاطري 😅'
AFTER:  '[MENTION] يعطيك الصحة برد